# UniCompot -- Arabic RAG Pipeline + Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DrAliAliedani/UniCompot/blob/main/notebooks/UniCompot_RAG_and_Evaluation.ipynb)


**Requirements before running:**
1. A GPU runtime (Colab: `Runtime > Change runtime type > GPU`). A 4-bit
   8B-parameter model + an E5-large embedder + a BGE reranker + an mDeBERTa
   NLI model all resident at once is tight on a T4 (16 GB) -- an L4 or A100
   runtime (Colab Pro/Pro+) is more comfortable and much faster for the
   evaluation loops below.
2. A Hugging Face token with access to `inceptionai/Jais-2-8B-Chat` (accept
   the model's license on its Hugging Face page first), stored as a Colab
   secret named `HF_TOKEN` (key icon in the left sidebar, then grant this
   notebook access).
3. `allfile.pdf` uploaded to `/content/` (drag it into the Colab file
   pane), or adjust the path in the loader cell if you mount Google Drive
   instead.

Run the cells top to bottom. The Gradio cell is optional (interactive demo
only) -- skip it if you just want the evaluation results; it blocks the
notebook with `debug=True` until you stop it manually.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers langchain langchain-community faiss-gpu-cu12 rank-bm25
!pip install -q pypdf
!pip install -q -U FlagEmbedding gradio openpyxl
# NOTE: the original script also installed `g4f` (an unofficial free-GPT
# wrapper) -- it is never imported or used anywhere in the pipeline, so it
# has been dropped here. `openpyxl` and `gradio`/`FlagEmbedding` were
# missing from the original install cell even though `df.to_excel(...)`,
# `import gradio`, and `from FlagEmbedding import FlagReranker` are used
# later in the script -- added them so the notebook runs on a fresh runtime
# without a mid-run ModuleNotFoundError.


## 2. Hugging Face login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token=token)


## 3. Load the Jais-2-8B-Chat model (4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "inceptionai/Jais-2-8B-Chat"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


## 4. Load the embedding model

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Best for Arabic: intfloat/multilingual-e5-large
embedding_model_name = "intfloat/multilingual-e5-large"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)


## 5. Canonical category/keyword table -- bug fix

**Bug found while combining the two files:** the original script tagged each
*document chunk* at indexing time with one hand-written Arabic/English
keyword dictionary (in the `# --- ADD APPROPRIATE METADATA DYNAMICALLY ---`
cell), and routed each *query* at question time with a **second, separate**
dictionary (`auto_categorize_query`). The two had drifted apart -- the query
router recognised words like `نساء`, `طالبة`, `حضانة`, `بلاستيك`, `استدامة`,
`موظف`, `حقوق`, `عامل`, `غياب`, `غش`, `طالب` that the indexing-time tagger
never used, and the indexing tagger used English keywords (`women`,
`discrimination`, `waste`, `environment`, ...) that a query router working on
Arabic questions would never see. A query could therefore get routed to a
metadata category that few or none of the chunks were actually tagged with,
silently hurting retrieval.

**Fix:** one canonical `CATEGORY_KEYWORDS` table, used for both indexing
(`categorize_text`) and query routing (`route_query`), so a query can only
ever be routed to a category documents were actually tagged with.


In [ ]:
CATEGORY_KEYWORDS = {
    "شؤون المرأة والمساواة": [
        "المرأة", "نساء", "تمكين", "الامومة", "الأمومة", "التمييز",
        "طالبة", "حضانة",
    ],
    "البيئة والاستدامة": [
        "النفايات", "تدوير", "المياه", "البيئة", "استدامة", "بلاستيك",
    ],
    "الحرية الأكاديمية وحقوق العاملين": [
        "الحرية", "الاكاديمية", "الأكاديمية", "العمال", "عامل", "موظف",
        "حقوق",
    ],
    "شؤون الطلبة والامتحانات": [
        "امتحان", "الطلبة", "طالب", "عقوبات", "ترقين", "الفصل", "غياب",
        "غش",
    ],
}
DEFAULT_CATEGORY = "عام"      # used for chunks that match no category
ROUTE_ALL_LABEL = "الكل"      # "search everything" route

# Single source of truth for tagging a DOCUMENT CHUNK at indexing time.
def categorize_text(text, fallback=DEFAULT_CATEGORY):
    t = text.lower()
    for category, kws in CATEGORY_KEYWORDS.items():
        if any(kw in t for kw in kws):
            return category
    return fallback

# Single source of truth for routing a QUERY at question time. Uses the
# exact same CATEGORY_KEYWORDS table as categorize_text(), so a query can
# only ever be routed to a category documents were actually tagged with.
def route_query(query, fallback=ROUTE_ALL_LABEL):
    t = query.lower()
    for category, kws in CATEGORY_KEYWORDS.items():
        if any(kw in t for kw in kws):
            return category
    return fallback

print("Canonical CATEGORY_KEYWORDS table ready.")


## 6. Load the PDF, chunk it, tag chunks, build BM25 + FAISS indexes

In [ ]:
import faiss
import numpy as np
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from rank_bm25 import BM25Okapi
import re

loader = PyPDFLoader("/content/allfile.pdf")
arabic_text_raw_documents = loader.load()

docs = []

if not arabic_text_raw_documents:
    print("Error: No documents loaded from PDF.")
else:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,   # ~300-400 words, usually a full page/full law
        chunk_overlap=300, # keeps headers attached to their text
        separators=["\n\n", "\n", ".", "؟", "!", " "]
    )
    docs = text_splitter.split_documents(arabic_text_raw_documents)

    if not docs:
        print("Error: No chunks created after splitting documents.")

if docs:
    # FIX: use the single canonical categorize_text() instead of a second,
    # hand-maintained if/elif chain (see the bug note above).
    for doc in docs:
        doc.metadata["category"] = categorize_text(doc.page_content)

    texts = [doc.page_content for doc in docs]

    def tokenize_arabic(text):
        return re.findall(r"\w+", text)
    tokenized_corpus = [tokenize_arabic(text) for text in texts]
    bm25 = BM25Okapi(tokenized_corpus)

    # E5 requires the "passage: " prefix on indexed text
    prefixed_texts = [f"passage: {text}" for text in texts]
    vectors = embeddings.embed_documents(prefixed_texts)
    vectors = np.array(vectors).astype("float32")
    dimension = vectors.shape[1]

    # HNSW parameters
    M = 64
    efConstruction = 200
    efSearch = 64
    index = faiss.IndexHNSWFlat(dimension, M)
    index.hnsw.efConstruction = efConstruction
    index.hnsw.efSearch = efSearch
    index.add(vectors)

    docstore = InMemoryDocstore({str(i): docs[i] for i in range(len(docs))})
    index_to_docstore_id = {i: str(i) for i in range(len(docs))}

    vector_db = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=docstore,
        index_to_docstore_id=index_to_docstore_id
    )

    print(f"FAISS HNSW ready with {len(docs)} chunks (dynamic metadata via categorize_text)")
    print("BM25 keyword search ready")
else:
    print("FAISS HNSW setup skipped due to no documents or chunks.")


## 7. Question generation + ad hoc dataset builder

In [ ]:
import random
import torch

def generate_question(text):
    system_prompt_gen_question = "أنت مساعد ذكي مهمتك إنشاء سؤال واحد واضح ومباشر يمكن الإجابة عليه من النص التالي."
    user_prompt_gen_question = f"النص:\n{text[:1000]}\n\nبناءً على النص أعلاه، أنشئ سؤالاً واحدًا:"

    messages_gen_question = [
        {"role": "system", "content": system_prompt_gen_question},
        {"role": "user", "content": user_prompt_gen_question},
    ]

    chat_text_gen_question = tokenizer.apply_chat_template(messages_gen_question, tokenize=False, add_generation_prompt=True)
    inputs_gen_question = tokenizer(chat_text_gen_question, return_tensors="pt").to(model.device)
    inputs_gen_question.pop("token_type_ids", None)

    with torch.no_grad():
        outputs_gen_question = model.generate(
            **inputs_gen_question,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    question = tokenizer.decode(outputs_gen_question[0][inputs_gen_question["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return question

# Ad hoc / exploratory question set (unstratified random sample). Kept for
# compatibility with earlier exploration; FIXED_EVAL_SET (section 12 below)
# is the stratified, seeded set to report results on.
def build_dataset(num_samples=100):
    dataset = []
    sampled_indices = random.sample(range(len(docs)), min(num_samples, len(docs)))
    for idx in sampled_indices:
        text = docs[idx].page_content
        query = generate_question(text)
        dataset.append({"query": query, "relevant_docs": [idx]})
    return dataset

print("Question generation and dataset building functions are ready.")


## 8. Reranker + hybrid search (with reranking)

In [ ]:
from FlagEmbedding import FlagReranker

# Lightweight reranker (fast + good)
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)

# Hybrid search + Cross-Encoder reranking.
def hybrid_search_with_rerank(query, k=6, alpha=0.5, category_filter="الكل"):
    initial_k = k * 4  # bigger pool for reranking

    filter_kwargs = {"category": category_filter} if category_filter != "الكل" else {}

    semantic_results = vector_db.similarity_search_with_score(
        f"query: {query}", k=initial_k, filter=filter_kwargs
    )

    tokenized_query = tokenize_arabic(query)
    keyword_scores = bm25.get_scores(tokenized_query)

    valid_indices = [
        i for i, d in enumerate(docs)
        if category_filter == "الكل" or d.metadata.get("category") == category_filter
    ]
    for i in range(len(keyword_scores)):
        if i not in valid_indices:
            keyword_scores[i] = -9999

    keyword_indices = np.argsort(keyword_scores)[-initial_k:][::-1]

    combined_candidates = {}

    if semantic_results:
        scores = [s for _, s in semantic_results]
        max_s, min_s = max(scores), min(scores)
        range_s = max_s - min_s if max_s > min_s else 1
        for doc, score in semantic_results:
            for idx, d in enumerate(docs):
                if d.page_content == doc.page_content:
                    norm = 1 - ((score - min_s) / range_s)
                    combined_candidates[idx] = alpha * norm
                    break

    valid_scores = [s for s in keyword_scores if s > -9999]
    if valid_scores:
        max_k, min_k = max(valid_scores), min(valid_scores)
        range_k = max_k - min_k if max_k > min_k else 1
        for idx in keyword_indices:
            if keyword_scores[idx] == -9999:
                continue
            norm = (keyword_scores[idx] - min_k) / range_k
            if idx in combined_candidates:
                combined_candidates[idx] += (1 - alpha) * norm
            else:
                combined_candidates[idx] = (1 - alpha) * norm

    top_candidates = sorted(combined_candidates.items(), key=lambda x: x[1], reverse=True)[:initial_k]
    candidate_docs = [docs[idx] for idx, _ in top_candidates]

    pairs = [(query, doc.page_content) for doc in candidate_docs]
    #rerank_scores = reranker.predict(pairs)
    rerank_scores = reranker.compute_score(pairs)
    reranked = sorted(zip(candidate_docs, rerank_scores), key=lambda x: x[1], reverse=True)
    final_docs = [doc for doc, _ in reranked[:k]]
    return final_docs


## 9. Hybrid search (no reranking) -- moved up

**Bug found:** in the original script, `get_context()` (below) calls
`hybrid_search_no_rerank(...)` whenever `use_reranker=False`, but
`hybrid_search_no_rerank` itself wasn't *defined* until much later in the
file (after the Gradio app, the dataset builder, and several evaluation
cells). Any call to `get_context(..., use_reranker=False)` before that point
would raise `NameError: name 'hybrid_search_no_rerank' is not defined`. It's
moved here, immediately before `get_context`, so both branches are defined
before they're used.


In [ ]:
# Hybrid search without Cross-Encoder reranking -- combines semantic and
# keyword scores directly, no final reranking step.
def hybrid_search_no_rerank(query, k=6, alpha=0.5, category_filter="الكل"):
    initial_k = k * 4

    filter_kwargs = {"category": category_filter} if category_filter != "الكل" else {}

    semantic_results = vector_db.similarity_search_with_score(
        f"query: {query}", k=initial_k, filter=filter_kwargs
    )

    tokenized_query = tokenize_arabic(query)
    bm25_scores = bm25.get_scores(tokenized_query)

    valid_indices = [
        i for i, d in enumerate(docs)
        if category_filter == "الكل" or d.metadata.get("category") == category_filter
    ]
    for i in range(len(bm25_scores)):
        if i not in valid_indices:
            bm25_scores[i] = -9999

    keyword_indices = np.argsort(bm25_scores)[-initial_k:][::-1]

    combined_candidates = {}

    if semantic_results:
        scores = [s for _, s in semantic_results]
        max_s, min_s = max(scores), min(scores)
        range_s = max_s - min_s if max_s > min_s else 1
        for doc, score in semantic_results:
            for idx, d in enumerate(docs):
                if d.page_content == doc.page_content:
                    norm = 1 - ((score - min_s) / range_s) if range_s > 0 else 0.0
                    combined_candidates[idx] = alpha * norm
                    break

    valid_bm25_scores = [s for s in bm25_scores if s > -9999]
    if valid_bm25_scores:
        max_k, min_k = max(valid_bm25_scores), min(valid_bm25_scores)
        range_k = max_k - min_k if max_k > min_k else 1
        for idx in keyword_indices:
            if bm25_scores[idx] == -9999:
                continue
            norm = (bm25_scores[idx] - min_k) / range_k if range_k > 0 else 0.0
            if idx in combined_candidates:
                combined_candidates[idx] += (1 - alpha) * norm
            else:
                combined_candidates[idx] = (1 - alpha) * norm

    top_candidates = sorted(combined_candidates.items(), key=lambda x: x[1], reverse=True)[:k]
    final_docs = [docs[idx] for idx, _ in top_candidates]
    return final_docs


## 10. get_context -- assembles the retrieved context

In [ ]:
def get_context(query, k=6, search_type="hybrid", alpha=0.5, category_filter="الكل", use_reranker=True):
    related_docs = []

    if search_type == "semantic":
        filter_kwargs = {"category": category_filter} if category_filter != "الكل" else {}
        related_docs = vector_db.similarity_search(f"query: {query}", k=k, filter=filter_kwargs)
    elif search_type == "keyword":
        if use_reranker:
            related_docs = hybrid_search_with_rerank(query, k=k, alpha=0.0, category_filter=category_filter)
        else:
            related_docs = hybrid_search_no_rerank(query, k=k, alpha=0.0, category_filter=category_filter)
    else:  # hybrid (default)
        if use_reranker:
            related_docs = hybrid_search_with_rerank(query, k=k, alpha=alpha, category_filter=category_filter)
        else:
            related_docs = hybrid_search_no_rerank(query, k=k, alpha=alpha, category_filter=category_filter)

    context_parts = []
    for i, doc in enumerate(related_docs):
        context_parts.append(f"--- [مقتطف {i+1}] ---\n{doc.page_content}")
    return "\n\n".join(context_parts)


## 11. NLI model (faithfulness / entailment scoring)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nli_model_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_classifier = AutoModelForSequenceClassification.from_pretrained(nli_model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_classifier.to(device)
nli_classifier.eval()


## 12. Relevance / faithfulness metrics

In [ ]:
import time
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

evaluation_history = {"faithfulness": [], "relevance": [], "latency": []}

# Relevance: question vs. answer.
def calculate_cosine_similarity(text1, text2, prefix1="query: ", prefix2="passage: "):
    if not text1.strip() or not text2.strip():
        return 0.0
    vec1 = embeddings.embed_query(f"{prefix1}{text1}")
    vec2 = embeddings.embed_query(f"{prefix2}{text2}")
    return max(0.0, float(cosine_similarity([vec1], [vec2])[0][0]))

# Evaluates each sentence of the answer against each excerpt of the context.
def calculate_true_faithfulness(context, answer):
    if not context.strip() or not answer.strip():
        return 0.0

    sentences = re.split(r"(?<=[.؟!\n])\s+", answer.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    excerpts = context.split("--- [مقتطف")
    excerpts = [e.strip() for e in excerpts if len(e.strip()) > 20]

    if not sentences or not excerpts:
        return 0.0

    total_score = 0.0
    id2label = nli_classifier.config.id2label

    for sentence in sentences:
        best_sentence_score = 0.0
        for excerpt in excerpts:
            inputs = nli_tokenizer(excerpt, sentence, truncation=True, max_length=512,
                                    return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = nli_classifier(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            entailment_prob = 0.0
            for idx, prob in enumerate(probs):
                if "entailment" in id2label[idx].lower():
                    entailment_prob = prob.item()
                    break
            if entailment_prob > best_sentence_score:
                best_sentence_score = entailment_prob
        total_score += best_sentence_score

    return float(total_score / len(sentences))

# Same computation as calculate_true_faithfulness(), but also returns the
# per-sentence entailment scores and a hallucination rate (fraction of
# sentences whose best entailment score is below `threshold`), with no
# extra model calls.
#
# FIX: the early-exit branch below originally returned a 2-item tuple
# (0.0, []) while the normal path returns 3 items, which raises a
# ValueError on unpacking (a, b, c = ...) whenever context/answer is
# empty. Both paths now return 3 items.
def calculate_true_faithfulness_with_detail(context, answer, threshold=0.5):
    if not context.strip() or not answer.strip():
        return 0.0, [], 0.0

    sentences = re.split(r"(?<=[.؟!\n])\s+", answer.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    excerpts = context.split("--- [مقتطف")
    excerpts = [e.strip() for e in excerpts if len(e.strip()) > 20]
    if not sentences or not excerpts:
        return 0.0, [], 0.0

    per_sentence = []
    id2label = nli_classifier.config.id2label
    for sentence in sentences:
        best = 0.0
        for excerpt in excerpts:
            inputs = nli_tokenizer(excerpt, sentence, truncation=True, max_length=512,
                                    return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = nli_classifier(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            for idx, prob in enumerate(probs):
                if "entailment" in id2label[idx].lower():
                    best = max(best, prob.item())
                    break
        per_sentence.append(best)

    faithfulness = sum(per_sentence) / len(per_sentence)
    hallucination_rate = sum(s < threshold for s in per_sentence) / len(per_sentence)
    return faithfulness, per_sentence, hallucination_rate

print("Relevance/faithfulness metric functions ready.")


## 13. ask_jais -- the full RAG answer + evaluation call

In [ ]:
def ask_jais(question, search_type="hybrid", alpha=0.5, category_filter="الكل", use_reranker=True):
    start_time = time.time()

    context = get_context(question, search_type=search_type, alpha=alpha,
                           category_filter=category_filter, use_reranker=use_reranker)

    system_prompt = "أنت مساعد ذكي يختص ياجابة اسئلة الطلبة وتوضيح تفاصيل النصوص الجامعية. استخدم القوانين والضوابط التالية للإجابة على سؤال الطالب بدقة."
    user_prompt = f"السياق:\n{context}\n\nالسؤال: {question}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    inputs.pop("token_type_ids", None)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=0.3, top_p=0.9, pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    latency = time.time() - start_time

    raw_relevance = calculate_cosine_similarity(question, answer, prefix1="query: ", prefix2="passage: ")
    raw_faithfulness = calculate_true_faithfulness(context, answer)

    rel_percentage = raw_relevance * 100
    faith_percentage = raw_faithfulness * 100

    evaluation_history["latency"].append(latency)
    evaluation_history["relevance"].append(raw_relevance)
    evaluation_history["faithfulness"].append(raw_faithfulness)

    return answer, round(latency, 2), round(faith_percentage, 2), round(rel_percentage, 2)

print("ask_jais ready.")


## 14. Closed-book (no-RAG) baseline

In [ ]:
# Calls the LLM directly with no retrieved context, for the non-RAG baseline
# requested in review. Faithfulness is undefined without a context, so score
# these answers with the human-evaluation sheet (section 19) instead,
# following closed-book LM evaluation practice (Roberts, Raffel & Shazeer,
# EMNLP 2020).
def ask_jais_closed_book(question):
    start_time = time.time()
    system_prompt = ("أنت مساعد ذكي يختص بإجابة اسئلة الطلبة حول أنظمة الجامعات العراقية. "
                      "أجب من معرفتك العامة إن لم يتوفر لديك سياق.")
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": question}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    inputs.pop("token_type_ids", None)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, do_sample=True,
                                  temperature=0.3, top_p=0.9,
                                  pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    latency = time.time() - start_time
    return answer, round(latency, 2)

print("ask_jais_closed_book ready.")

## 15. Citation-accuracy check

In [ ]:
import re as _re

ARTICLE_PATTERN = _re.compile(r"(?:المادة|الفقرة)\s*[\(\[]?\s*(\d+)")

# Fraction of article/clause numbers ("المادة N") mentioned in the ANSWER
# that also appear in the RETRIEVED CONTEXT -- a cheap, fully automatic
# proxy for regulatory citation grounding; complements, but does not
# replace, human correctness rating.
def citation_accuracy(context, answer):
    cited_in_answer = set(ARTICLE_PATTERN.findall(answer))
    if not cited_in_answer:
        return None
    cited_in_context = set(ARTICLE_PATTERN.findall(context))
    supported = cited_in_answer & cited_in_context
    return len(supported) / len(cited_in_answer)

print("citation_accuracy ready.")


## 16. (Optional) Gradio demo

Skip this cell if you only want the evaluation results below -- it launches
a public Gradio link and blocks the notebook (`debug=True`) until you
interrupt it.

**Fix applied:** the original `chatbot()` used a second hardcoded router,
`auto_categorize_query`, with its own drifted keyword list (the same bug as
section 5). It now calls `route_query()` -- the same canonical table used to
tag the corpus -- and the category dropdown is built from
`CATEGORY_KEYWORDS` directly so the UI can never list a category that isn't
actually one of the tagged ones.


In [ ]:
import gradio as gr

def chatbot(question, search_type, alpha_value, category_filter):
    if category_filter == "تلقائي (Auto)":
        actual_category = route_query(question)
    else:
        actual_category = category_filter
    ans, lat, avg_f, avg_r = ask_jais(question, search_type, alpha_value, actual_category)
    return ans, f"{lat} ثانية", f"{avg_f}%", f"{avg_r}%", f"تم البحث في قسم: {actual_category}"

category_choices = ["تلقائي (Auto)", ROUTE_ALL_LABEL] + list(CATEGORY_KEYWORDS.keys()) + [DEFAULT_CATEGORY]

with gr.Blocks() as interface:
    gr.Markdown("# جامعة البصرة - نظام البحث الهجين الذكي مع التقييم")
    gr.Markdown("اكتب سؤالك وسيقوم النظام بتوجيهه تلقائياً إلى القسم المناسب وتوليد إجابة دقيقة.")

    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(label="اكتب/ي سؤالك هنا")
            category_input = gr.Dropdown(choices=category_choices, label="تصنيف البحث (Metadata)", value="تلقائي (Auto)")
            search_type_input = gr.Radio(choices=["semantic", "keyword", "hybrid"], label="نوع البحث", value="hybrid")
            alpha_input = gr.Slider(minimum=0, maximum=1, value=0.5, step=0.1, label="نسبة البحث الدلالي")
            submit_btn = gr.Button("إرسال", variant="primary")

        with gr.Column(scale=3):
            routed_category_output = gr.Textbox(label="مسار البحث (Metadata)", lines=1)
            answer_output = gr.Textbox(label="إجابة النظام", lines=5)
            with gr.Row():
                latency_output = gr.Textbox(label="زمن الاستجابة (Latency)")
                faith_output = gr.Textbox(label="معدل الدقة (Avg Faithfulness)")
                rel_output = gr.Textbox(label="معدل الارتباط (Avg Relevance)")

    submit_btn.click(
        fn=chatbot,
        inputs=[question_input, search_type_input, alpha_input, category_input],
        outputs=[answer_output, latency_output, faith_output, rel_output, routed_category_output]
    )

# interface.launch(share=True, debug=True)   # uncomment to actually launch


## 17. Build the ad hoc exploratory dataset

**Bug found:** the original script called `dataset[:10]` / `dataset[:100]`
in several comparison cells long before `dataset = build_dataset()` was
ever executed (that line was the very last cell in the file) -- every one of
those comparison cells would raise `NameError: name 'dataset' is not
defined` if run in file order. `dataset` is built here, before anything
uses it. Note this is the old *unstratified* random sample kept only for
compatibility/exploration -- the evaluation from section 18 onward uses the
fixed, stratified `FIXED_EVAL_SET` instead, per the reviewers' request.


In [ ]:
dataset = build_dataset(num_samples=100)
print(f"Dataset generated with {len(dataset)} samples.")


## 18. Fixed, stratified evaluation set (replaces the 3 ad hoc lists)

The original notebook used three separate ad hoc query lists of
inconsistent size (5, 11, 10), making results non-comparable and the sample
too small. This builds one query set, sampled once, stratified
proportionally to the corpus's own category mix, with a fixed seed -- reused
for every metric and table below so results are directly comparable and the
experiment is reproducible.


In [ ]:
import random as _random

def build_fixed_eval_set(n_total=60, seed=42):
    _random.seed(seed)
    by_cat = {}
    for i, d in enumerate(docs):
        by_cat.setdefault(d.metadata.get("category", DEFAULT_CATEGORY), []).append(i)
    total_chunks = len(docs)
    eval_set = []
    for cat, idxs in by_cat.items():
        n_cat = max(1, round(n_total * len(idxs) / total_chunks))
        sample_idxs = _random.sample(idxs, min(n_cat, len(idxs)))
        for idx in sample_idxs:
            q = generate_question(docs[idx].page_content)
            eval_set.append({"query": q, "relevant_docs": [idx], "category": cat})
    _random.shuffle(eval_set)
    return eval_set[:n_total] if len(eval_set) > n_total else eval_set

# Tip: try n_total=10 first for a quick sanity check (each item costs one
# LLM call to auto-generate its question), then re-run with n_total=60 for
# the numbers you report.
import json

FIXED_EVAL_SET = build_fixed_eval_set(n_total=60, seed=42)
json.dump(FIXED_EVAL_SET, open("fixed_eval_set_n60_seed42.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(f"FIXED_EVAL_SET ready: {len(FIXED_EVAL_SET)} queries (n=60, seed=42). "
      f"Report this exact n and seed for reproducibility.")


## 19. Retrieval-only helper (isolates retrieval from generation)

In [ ]:
content_to_idx = {d.page_content: i for i, d in enumerate(docs)}  # O(1) lookup

# Ranked list of corpus indices (docs[i]) the system would retrieve for
# `query`, without calling the LLM -- mirrors get_context()'s branching so
# retrieval metrics reflect exactly what the deployed system returns.
def retrieve_ranked_indices(query, k=6, search_type="hybrid", alpha=0.5,
                             category_filter="الكل", use_reranker=True):
    if search_type == "semantic":
        filter_kwargs = {"category": category_filter} if category_filter != "الكل" else {}
        related_docs = vector_db.similarity_search(f"query: {query}", k=k, filter=filter_kwargs)
    elif search_type == "keyword":
        fn = hybrid_search_with_rerank if use_reranker else hybrid_search_no_rerank
        related_docs = fn(query, k=k, alpha=0.0, category_filter=category_filter)
    else:
        fn = hybrid_search_with_rerank if use_reranker else hybrid_search_no_rerank
        related_docs = fn(query, k=k, alpha=alpha, category_filter=category_filter)
    return [content_to_idx[d.page_content] for d in related_docs if d.page_content in content_to_idx]


## 20. Standard IR metrics: Recall@k, MRR@10, nDCG@10

Definitions follow Jarvelin & Kekalainen (2002) for (n)DCG and the TREC
convention for MRR (Voorhees, 1999).


In [ ]:
def recall_at_k(ranked_indices, relevant_set, k):
    if not relevant_set:
        return None
    topk = set(ranked_indices[:k])
    return len(topk & relevant_set) / len(relevant_set)

def reciprocal_rank(ranked_indices, relevant_set, k=10):
    for rank, idx in enumerate(ranked_indices[:k], start=1):
        if idx in relevant_set:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked_indices, relevant_set, k=10):
    def dcg(rels):
        return sum((2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(rels))
    gains = [1.0 if idx in relevant_set else 0.0 for idx in ranked_indices[:k]]
    ideal = sorted(gains, reverse=True)
    idcg = dcg(ideal)
    return dcg(gains) / idcg if idcg > 0 else 0.0

def evaluate_retrieval(eval_set, search_type="hybrid", alpha=0.5, use_reranker=True, k=10):
    r5s, r10s, mrrs, ndcgs = [], [], [], []
    for item in eval_set:
        relevant = set(item["relevant_docs"])
        if not relevant:
            continue
        ranked = retrieve_ranked_indices(item["query"], k=k, search_type=search_type,
                                          alpha=alpha, use_reranker=use_reranker)
        r5s.append(recall_at_k(ranked, relevant, 5))
        r10s.append(recall_at_k(ranked, relevant, 10))
        mrrs.append(reciprocal_rank(ranked, relevant, 10))
        ndcgs.append(ndcg_at_k(ranked, relevant, 10))
    return {
        "Recall@5": r5s, "Recall@10": r10s, "MRR@10": mrrs, "nDCG@10": ndcgs,
        "Recall@5_mean": float(np.mean(r5s)), "Recall@10_mean": float(np.mean(r10s)),
        "MRR@10_mean": float(np.mean(mrrs)), "nDCG@10_mean": float(np.mean(ndcgs)),
    }

print("Retrieval metric functions ready.")


## 21. Paired bootstrap significance testing (Koehn, 2004)

IR/NLG metrics are rarely normally distributed, so a paired bootstrap (or
randomisation) test is generally preferred to a t-test here (Smucker, Allan
& Carterette, 2007; Koehn, 2004).


In [ ]:
def paired_bootstrap_test(scores_a, scores_b, n_resamples=10000, seed=13):
    # scores_a, scores_b: paired per-query scores for system A and B (same
    # queries, same order). Returns mean_diff, 95% CI, and p for
    # H0: equal mean.
    rng = np.random.default_rng(seed)
    a = np.asarray(scores_a, dtype=float)
    b = np.asarray(scores_b, dtype=float)
    assert len(a) == len(b), "scores must be paired (same queries, same order)"
    diffs = a - b
    observed = diffs.mean()
    n = len(diffs)
    boot_means = np.empty(n_resamples)
    for i in range(n_resamples):
        sample = rng.choice(diffs, size=n, replace=True)
        boot_means[i] = sample.mean()
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    p_value = 2 * min((boot_means <= 0).mean(), (boot_means >= 0).mean())
    p_value = min(p_value, 1.0)
    return {"mean_diff": float(observed), "ci95_low": float(ci_low),
            "ci95_high": float(ci_high), "p_value": float(p_value), "n": n}

print("paired_bootstrap_test ready. Only call a difference significant in the paper when p < .05 here.")


## 22. Metadata-routing evaluation (accuracy + confusion matrix)

Every `FIXED_EVAL_SET` item carries the true category of the chunk its
question was generated from, so routing accuracy is simply: does
`route_query(query)` match that true category?


In [ ]:
from collections import Counter

def evaluate_routing(eval_set):
    y_true, y_pred = [], []
    for item in eval_set:
        y_true.append(item["category"])
        y_pred.append(route_query(item["query"]))
    correct = sum(t == p for t, p in zip(y_true, y_pred))
    accuracy = correct / len(eval_set)
    categories = sorted(set(y_true) | set(y_pred))
    confusion = {t: Counter() for t in categories}
    for t, p in zip(y_true, y_pred):
        confusion[t][p] += 1
    return {"accuracy": accuracy, "confusion": confusion, "n": len(eval_set)}

print("evaluate_routing ready.")


## 23. Human-evaluation export sheet + Cohen's kappa

Automatic metrics (NLI-faithfulness, cosine-relevance) are proxies, not
ground truth -- this pairs automatic proxy metrics with human validation,
the same design used by RAGAS (Es et al., EACL 2024). The rating sheet
follows the evaluation-sheet recommendations of Howcroft et al. (INLG
2020): clear closed rating scales with written anchors, one construct per
column, rater identity kept separate from item order.


In [ ]:
import pandas as pd
import uuid

RATING_ANCHORS = {
    "correctness": "1 = answer is wrong or irrelevant; 3 = partially correct; "
                    "5 = fully and precisely correct given the regulations.",
    "groundedness": "1 = answer states facts not supported by any official "
                     "regulation; 3 = partially supported; 5 = every claim is "
                     "directly supported by an identifiable regulation.",
    "completeness": "1 = misses the key point of the question; 3 = partially "
                     "answers it; 5 = fully answers everything the question asked.",
}

def build_human_eval_sheet(rag_results, closed_book_results=None, seed=7):
    rows = []
    for r in rag_results:
        rows.append({"item_id": str(uuid.uuid4())[:8], "system": "UniCompot (RAG)",
                     "question": r.get("Question", r.get("query")),
                     "context_shown_to_rater": r.get("Context", ""),
                     "answer": r.get("Answer", r.get("answer", ""))})
    if closed_book_results:
        for r in closed_book_results:
            rows.append({"item_id": str(uuid.uuid4())[:8], "system": "Closed-book (no RAG)",
                         "question": r["Question"], "context_shown_to_rater": "",
                         "answer": r["Answer"]})
    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)  # blind order
    for construct in RATING_ANCHORS:
        df[construct] = ""  # rater fills in 1-5
    df["rater_id"] = ""
    return df

def cohens_kappa(rater1_scores, rater2_scores, labels=(1, 2, 3, 4, 5)):
    n = len(rater1_scores)
    assert n == len(rater2_scores)
    po = sum(a == b for a, b in zip(rater1_scores, rater2_scores)) / n
    p1 = {l: sum(x == l for x in rater1_scores) / n for l in labels}
    p2 = {l: sum(x == l for x in rater2_scores) / n for l in labels}
    pe = sum(p1[l] * p2[l] for l in labels)
    return (po - pe) / (1 - pe) if pe != 1 else 1.0

print("Human-eval sheet + Cohen's kappa ready.")


## 24. Run: unified retrieval comparison table (search type x alpha x reranker)

In [ ]:
configs = [
    {"label": "Semantic",                  "search_type": "semantic", "alpha": 0.5, "use_reranker": True},
    {"label": "Keyword (BM25)",             "search_type": "keyword",  "alpha": 0.5, "use_reranker": True},
    {"label": "Hybrid alpha=0.3",           "search_type": "hybrid",   "alpha": 0.3, "use_reranker": True},
    {"label": "Hybrid alpha=0.5",           "search_type": "hybrid",   "alpha": 0.5, "use_reranker": True},
    {"label": "Hybrid alpha=0.7",           "search_type": "hybrid",   "alpha": 0.7, "use_reranker": True},
    {"label": "Hybrid alpha=0.5 (no rerank)", "search_type": "hybrid", "alpha": 0.5, "use_reranker": False},
]

comparison_rows = []
retrieval_raw_scores = {}  # per-query scores, kept for the significance test below

for cfg in configs:
    res = evaluate_retrieval(FIXED_EVAL_SET, search_type=cfg["search_type"],
                              alpha=cfg["alpha"], use_reranker=cfg["use_reranker"], k=10)
    retrieval_raw_scores[cfg["label"]] = res
    comparison_rows.append({
        "Configuration": cfg["label"],
        "Recall@5": round(res["Recall@5_mean"], 4),
        "Recall@10": round(res["Recall@10_mean"], 4),
        "MRR@10": round(res["MRR@10_mean"], 4),
        "nDCG@10": round(res["nDCG@10_mean"], 4),
        "n_queries": len(res["nDCG@10"]),
    })

df_unified_comparison = pd.DataFrame(comparison_rows)
display(df_unified_comparison)
df_unified_comparison.to_csv("unified_retrieval_comparison.csv", index=False, encoding="utf-8-sig")
print("Saved unified_retrieval_comparison.csv")


## 25. Run: full RAG evaluation over FIXED_EVAL_SET (chosen config)

In [ ]:
BEST_CONFIG = {"search_type": "hybrid", "alpha": 0.5, "use_reranker": True}  # set from the table above

rag_results = []
for item in FIXED_EVAL_SET:
    q = item["query"]
    context = get_context(q, search_type=BEST_CONFIG["search_type"], alpha=BEST_CONFIG["alpha"],
                           category_filter="الكل", use_reranker=BEST_CONFIG["use_reranker"])
    answer, latency, faith_pct, rel_pct = ask_jais(q, search_type=BEST_CONFIG["search_type"],
                                                    alpha=BEST_CONFIG["alpha"], category_filter="الكل",
                                                    use_reranker=BEST_CONFIG["use_reranker"])
    _, per_sentence, hallucination_rate = calculate_true_faithfulness_with_detail(context, answer)
    cite_acc = citation_accuracy(context, answer)
    rag_results.append({
        "Question": q, "Context": context, "Answer": answer,
        "Latency (s)": latency, "Faithfulness (%)": faith_pct, "Relevance (%)": rel_pct,
        "Hallucination Rate": round(hallucination_rate, 3),
        "Citation Accuracy": None if cite_acc is None else round(cite_acc, 3),
    })

df_rag_results = pd.DataFrame(rag_results)
display(df_rag_results)
df_rag_results.to_csv("rag_full_eval_results.csv", index=False, encoding="utf-8-sig")
print(f"Mean faithfulness: {df_rag_results['Faithfulness (%)'].mean():.1f}%")
print(f"Mean relevance:    {df_rag_results['Relevance (%)'].mean():.1f}%")
print(f"Mean hallucination rate: {df_rag_results['Hallucination Rate'].mean():.1%}")


## 26. Run: closed-book baseline over FIXED_EVAL_SET

In [ ]:
closed_book_results = []
for item in FIXED_EVAL_SET:
    ans, lat = ask_jais_closed_book(item["query"])
    closed_book_results.append({"Question": item["query"], "Answer": ans, "Latency (s)": lat})

df_closed_book = pd.DataFrame(closed_book_results)
display(df_closed_book)
df_closed_book.to_csv("closed_book_baseline_results.csv", index=False, encoding="utf-8-sig")


## 27. Run: routing accuracy + confusion matrix

In [ ]:
routing_results = evaluate_routing(FIXED_EVAL_SET)
print(f"Routing accuracy: {routing_results['accuracy']:.1%} (n={routing_results['n']})")
for true_cat, preds in routing_results["confusion"].items():
    print(f"{true_cat:35s} -> {dict(preds)}")


## 28. Run: significance test example (Hybrid vs. Semantic-only, nDCG@10)

In [ ]:
sig = paired_bootstrap_test(
    retrieval_raw_scores["Hybrid alpha=0.5"]["nDCG@10"],
    retrieval_raw_scores["Semantic"]["nDCG@10"],
)
print("Hybrid (alpha=0.5) vs Semantic-only, nDCG@10:")
print(f"  mean diff = {sig['mean_diff']:.4f}  95% CI [{sig['ci95_low']:.4f}, {sig['ci95_high']:.4f}]  "
      f"p = {sig['p_value']:.4f}  (n={sig['n']})")
print("Only call this difference significant in the paper if p < .05.")


## 29. Run: human-evaluation sheet export

**Gap found and fixed:** the original comment said the sheet's `system`
column would be "hidden from raters", but the export code never actually
dropped it. Fixed here -- the rater-facing file has no `system` column, and
a separate answer key (`item_id` -> `system`) is kept for you to score
against later, without ever handing it to the raters.


In [ ]:
sheet = build_human_eval_sheet(rag_results=rag_results, closed_book_results=closed_book_results, seed=7)

rater_facing = sheet.drop(columns=["system"])
rater_facing.to_excel("human_eval_sheet_BLIND.xlsx", index=False)
sheet[["item_id", "system"]].to_csv("human_eval_answer_key.csv", index=False)  # keep for yourself only

print(f"Exported human_eval_sheet_BLIND.xlsx ({len(rater_facing)} rows, blinded, no system column) "
      f"and human_eval_answer_key.csv (for your own scoring, do not share with raters).")
print("Have two independent raters unfamiliar with the system score it (correctness/groundedness/"
      "completeness, 1-5), then: cohens_kappa(rater1['correctness'], rater2['correctness'])")


## 30. Figure export notes

Replace any `plt.savefig(...)` calls used for paper figures with a vector +
high-resolution pair:

```python
plt.savefig("fig_name.pdf", bbox_inches="tight")           # vector, for the camera-ready PDF
plt.savefig("fig_name.png", dpi=300, bbox_inches="tight")  # raster fallback, >=300 dpi
```

and embed the `.pdf` (or `.eps`) version in the LaTeX/Word source rather
than a low-DPI screen-resolution PNG.

## References for the methodology used above

- Jarvelin, K., & Kekalainen, J. (2002). Cumulated gain-based evaluation of
  IR techniques. *ACM Transactions on Information Systems*, 20(4), 422-446.
- Voorhees, E. M. (1999). The TREC-8 Question Answering Track Report.
  *Proceedings of TREC-8*.
- Koehn, P. (2004). Statistical significance tests for machine translation
  evaluation. *Proceedings of EMNLP 2004*.
- Smucker, M. D., Allan, J., & Carterette, B. (2007). A comparison of
  statistical significance tests for information retrieval evaluation.
  *Proceedings of CIKM 2007*.
- Es, S., James, J., Espinosa-Anke, L., & Schockaert, S. (2024). RAGAS:
  Automated evaluation of retrieval augmented generation. *Proceedings of
  the 18th Conference of the EACL: System Demonstrations*.
- Howcroft, D. M., et al. (2020). Twenty years of confusion in human
  evaluation: NLG needs an evaluation sheet and standard. *Proceedings of
  INLG 2020*.
- Roberts, A., Raffel, C., & Shazeer, N. (2020). How much knowledge can you
  pack into the parameters of a language model? *Proceedings of EMNLP
  2020*.
